# Narrator — ElevenLabs TTS Generator

Generates narration audio for each scene using the ElevenLabs API.
A YAML cache prevents redundant API calls: audio is regenerated only when
the narration text, voice, or model changes.

**Voice**: Jack  
**Model**: `eleven_v3` (expressive, audio tags supported)  
**Output**: `audio/scene_NN.mp3`  
**Cache**: `audio/narrator_cache.yaml`

Requires `ELEVENLABS_API_KEY` environment variable.

In [ ]:
import hashlib
import os
from datetime import datetime, timezone
from pathlib import Path

import yaml
from elevenlabs.client import ElevenLabs

AUDIO_DIR = Path("audio")
CACHE_FILE = AUDIO_DIR / "narrator_cache.yaml"
MODEL_ID = "eleven_v3"

# ── Voice configuration ────────────────────────────────────────────────────
# Find your voice ID at: https://elevenlabs.io/app/voice-lab
#   1. Open a voice → click "ID" next to the voice name → copy
# Premade voices (always available):
#   Rachel  21m00Tcm4TlvDq8ikWAM  (neutral, clear)
#   Josh    TxGEqnHWrfWFTfGW9XjX  (deep male)
#   Adam    pNInz6obpgDQGcFmaJgB  (narration)
VOICE_NAME = "Jack"
VOICE_ID   = "RL2gbGArFsmr05q4aJLj"   # ← paste your voice ID here
# ──────────────────────────────────────────────────────────────────────────

# ── Scene filter ───────────────────────────────────────────────────────────
# Generate only specific scenes to control credit usage.
# Set to None to generate all scenes.
# Example: SCENES_TO_GENERATE = ["scene_01", "scene_02"]
SCENES_TO_GENERATE: list[str] | None = None
# ──────────────────────────────────────────────────────────────────────────

AUDIO_DIR.mkdir(exist_ok=True)

client = ElevenLabs(api_key=os.environ["ELEVENLABS_API_KEY"])
print(f"Ready. Voice: {VOICE_NAME} ({VOICE_ID}), Model: {MODEL_ID}")

In [ ]:
NARRATIONS = {
    "scene_01": {
        "start": "0:00",
        "end": "1:30",
        "text": (
            "[excited] Rust is fast. "
            "Memory-safe, type-checked at compile time, fearless about concurrency. "
            "But the moment you try to build a real web application in Rust, things get rough. "
            "Router, ORM, auth, OpenAPI, migrations — "
            "you end up assembling them yourself, crate by crate. "
            "Meanwhile in Django, one command gives you ORM, admin, auth, and more. "
            "[awe] What if we could keep Rust's performance — and get back Django's productivity?"
        ),
    },
    "scene_02": {
        "start": "1:30",
        "end": "2:30",
        "text": (
            "[excited] That's why we built Reinhardt. "
            "A framework that fuses Django's philosophy with Rust's type safety. "
            "Its defining trait is a polylithic architecture. "
            "Use the full stack, or pull in just the ORM, or just the DI container — your call. "
            "Today, we'll build a full-stack app in fifteen minutes — REST API plus a WASM single-page frontend. "
            "Model, migrations, API, a reactive UI, admin, authentication — all of it."
        ),
    },
    "scene_03": {
        "start": "2:30",
        "end": "4:00",
        "text": (
            "Let's start by scaffolding the project. "
            "If you know Django, startproject will feel right at home — but this one takes a with-pages flag. "
            "One command gives you a manager script, a config module, an app directory, and a frontend module that compiles to WebAssembly. "
            "Next, add a posts app. startapp separates models, views, and serializers into their own files from day one. "
            "Open it in the editor. The backend tree looks like Django; the frontend folder holds pages, components, and the reactive router. "
            "A single features equals full, frontend in Cargo.toml pulls in ORM, admin, auth, and the WASM runtime — every battery, pre-charged. "
            "Drop a local.toml for the database URL and token signing key — environment-specific values that never hit version control."
        ),
    },
    "scene_04": {
        "start": "4:00",
        "end": "5:30",
        "text": (
            "Next, the model. "
            "Tag a struct with the model attribute and Reinhardt treats it as an ORM entity. "
            "Per-field constraints go in the field attribute — max length, primary key, auto timestamps. The knobs mirror Django's field options. "
            "[dramatic tone] The important part: every one of those constraints is checked at compile time. "
            "The macro synthesizes Post new, Post objects, and friends. All type-safe, all static. "
            "Domain methods live in ordinary impl blocks. No magic — just code you can read. "
            "The User model uses the user macro — one attribute replaces the entire impl BaseUser block. Hasher and username field, declared once."
        ),
    },
    "scene_05": {
        "start": "5:30",
        "end": "6:30",
        "text": (
            "With the model in place, migrations are next. "
            "makemigrations compares the model against the migration history and emits the delta as a Rust file. "
            "Because the output is Rust, you can review it with a regular git diff. No raw SQL strings to eyeball. "
            "migrate applies the schema to the database for real. "
            "[awe] The Django experience — but now statically typed end to end."
        ),
    },
    "scene_06": {
        "start": "6:30",
        "end": "8:00",
        "text": (
            "Now the serializer — the shape of the API. "
            "A plain Rust struct plus serde is the contract — PostSerializer doubles as the request body and the response body. "
            "Read-only fields? Skip deserializing. Conceptually it's DRF's ModelSerializer. "
            "The difference is that this mapping is checked at compile time. "
            "Then the ViewSet. The reinhardt viewset attribute wraps a builder for ModelViewSet, with pagination, filtering, and ordering as one-liners. "
            "That's it — list, retrieve, create, update, and destroy are all generated. "
            "[awe] About twenty lines, a full CRUD API."
        ),
    },
    "scene_07": {
        "start": "8:00",
        "end": "10:30",
        "text": (
            "Ten minutes in, the JSON API is live. Now the UI. "
            "Reinhardt's pages module compiles to WebAssembly — same crate, same types. "
            "Server data is exposed through the server function attribute — an async function that becomes a typed RPC the frontend can call directly. "
            "On the client, use_action turns that RPC into a reactive resource and re-renders when the data arrives. "
            "The page macro is the view DSL — braces, child blocks, and plain Rust expressions. No JSX. "
            "PostSerializer is the contract. Backend and frontend share the same Rust type — no schema drift, ever. "
            "runserver with-pages boots the API and the WASM bundler in one step. "
            "[excited] Open the browser and the list is rendering — end-to-end, fully typed."
        ),
    },
    "scene_08": {
        "start": "10:30",
        "end": "12:00",
        "text": (
            "And there's still the crown jewel: admin. "
            "Attach the admin attribute to a model and the admin page exists. "
            "createsuperuser — same command, same spirit as Django. "
            "[awe] Open /admin/ in the browser, and there it is. "
            "List, search, edit, delete — everything works. "
            "The list_display columns appear exactly as declared. "
            "Total hand-written code: ... a few lines."
        ),
    },
    "scene_09": {
        "start": "12:00",
        "end": "13:30",
        "text": (
            "Before the code, one picture. "
            "The inject attribute marks each argument as a dependency. "
            "At compile time, Reinhardt walks a provider registry and wires them in. "
            "No runtime reflection. No hidden globals. [pause] "
            "Now the code. "
            "AuthSettings is a settings fragment composed into ProjectSettings — token auth turned on, no globals. "
            "PostRepository is registered with the injectable factory attribute. One async function describes how to build it; the container does the rest. "
            "PublishInput carries a validated id — the range validator keeps invalid calls from ever reaching the database. "
            "pre_validate equals true on the server function attribute calls validate automatically. No boilerplate, no forgotten checks. "
            "Guard IsActiveUser is injected alongside the user — inactive accounts get 403 before the body runs. "
            "No token: 401. Inactive user: 403. Valid active token: 200. "
            "[dramatic tone] Declarative to write, statically enforced. That's the Reinhardt way."
        ),
    },
    "scene_10": {
        "start": "13:30",
        "end": "15:00",
        "text": (
            "[awe] Fifteen minutes. That's what we just did. "
            "Model, migrations, CRUD, WASM UI, admin, auth, DI. "
            "About eighty lines of Rust — and the frontend and backend share types. "
            "Wiring the same stack out of Axum, sea-orm, utoipa, and a separate frontend framework would take hundreds of lines — and a lot of design decisions. "
            "Django's productivity, with Rust's performance. Full stack. That's what Reinhardt is aiming at. "
            "cargo install reinhardt-admin-cli — you can start today. "
            "[excited] See you next time."
        ),
    },
}

In [ ]:
def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


def load_cache() -> dict:
    if CACHE_FILE.exists():
        with CACHE_FILE.open() as f:
            return yaml.safe_load(f) or {}
    return {}


def save_cache(cache: dict) -> None:
    with CACHE_FILE.open("w") as f:
        yaml.dump(cache, f, default_flow_style=False, allow_unicode=True)


def generate_audio(scene_id: str, scene: dict, cache: dict) -> str:
    """Generate audio for one scene; skip when text/voice/model are unchanged."""
    text = scene["text"]
    cache_key = text_hash(f"{text}|{VOICE_ID}|{MODEL_ID}")
    output_path = AUDIO_DIR / f"{scene_id}.mp3"

    cached = cache.get(scene_id, {})
    if cached.get("cache_key") == cache_key and output_path.exists():
        return "skipped"

    audio_bytes = client.text_to_speech.convert(
        voice_id=VOICE_ID,
        text=text,
        model_id=MODEL_ID,
        output_format="mp3_44100_128",
    )
    output_path.write_bytes(b"".join(audio_bytes))

    cache[scene_id] = {
        "cache_key": cache_key,
        "voice_id": VOICE_ID,
        "voice_name": VOICE_NAME,
        "model_id": MODEL_ID,
        "output_file": str(output_path),
        "start": scene["start"],
        "end": scene["end"],
        "generated_at": datetime.now(timezone.utc).isoformat(),
    }
    return "generated"


# ── Build target list ─────────────────────────────────────────────────────
cache = load_cache()
targets = {
    sid: scene
    for sid, scene in NARRATIONS.items()
    if SCENES_TO_GENERATE is None or sid in SCENES_TO_GENERATE
}

# Credit estimate (1 credit ≈ 1 character for eleven_turbo_v2_5)
# Only counts scenes that would actually be generated (not cached).
to_generate = {
    sid: scene for sid, scene in targets.items()
    if cache.get(sid, {}).get("cache_key") !=
       text_hash(f"{scene['text']}|{VOICE_ID}|{MODEL_ID}")
    or not (AUDIO_DIR / f"{sid}.mp3").exists()
}
est_credits = sum(len(s["text"]) for s in to_generate.values())
print(f"Scenes targeted : {len(targets)}")
print(f"To generate     : {len(to_generate)}  (cached/skipped: {len(targets) - len(to_generate)})")
print(f"Est. credits    : ~{est_credits}")
print()

# ── Run ───────────────────────────────────────────────────────────────────
results = {}
for scene_id, scene in targets.items():
    status = generate_audio(scene_id, scene, cache)
    results[scene_id] = status
    icon = "✓" if status == "skipped" else "🎙"
    print(f"{icon} {scene_id} ({scene['start']}–{scene['end']}): {status}")

save_cache(cache)

# ── Summary ───────────────────────────────────────────────────────────────
n_gen  = sum(1 for v in results.values() if v == "generated")
n_skip = sum(1 for v in results.values() if v == "skipped")
print(f"\nGenerated: {n_gen}  Skipped: {n_skip}")
print()
for mp3 in sorted(AUDIO_DIR.glob("scene_*.mp3")):
    print(f"  {mp3.name}  ({mp3.stat().st_size / 1024:.1f} KB)")